# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [25]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns

In [26]:
from sklearn.model_selection import train_test_split 

In [48]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier # por defecto este hace soft vouting, pilla los porcentajes de votaciones y predice
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, KFold

In [28]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [43]:
from xgboost import XGBClassifier

In [30]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import GradientBoostingClassifier
import xgboost
from sklearn.metrics import accuracy_score, classification_report

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [31]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columnas = [
    "preg",
    "plas",
    "pres",
    "skin",
    "test",
    "mass",
    "pedi",
    "age",
    "class",
]

df_diabetes = pd.read_csv(url, names=columnas)

print(df_diabetes.shape)
print(df_diabetes.head())

(768, 9)
   preg  plas  pres  skin  test  mass   pedi  age  class
0     6   148    72    35     0  33.6  0.627   50      1
1     1    85    66    29     0  26.6  0.351   31      0
2     8   183    64     0     0  23.3  0.672   32      1
3     1    89    66    23    94  28.1  0.167   21      0
4     0   137    40    35   168  43.1  2.288   33      1


In [32]:
df_diabetes.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   test    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   class   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [33]:
df_diabetes.describe()

,preg,plas,pres,skin,test,mass,pedi,age,class
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [50]:
X = df_diabetes.drop(columns=["class"])
y = df_diabetes["class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [35]:
modelo_bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    random_state=42,
    n_jobs=-1, # le dice que use todos los recursoso del ordenador, con -2 te da más recursos pa otra cosa
)

# 2. Validación cruzada con 10 K-Folds sobre los datos de entrenamiento
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    modelo_bagging, X_train, y_train, cv=kfold, scoring="accuracy"
)

print("Accuracy por fold (10 K-Folds):", cv_scores.round(4))
print(f"Accuracy media en CV: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# 3. Entrenar el modelo final en todo X_train y evaluar en X_test
modelo_bagging.fit(X_train, y_train)
y_pred = modelo_bagging.predict(X_test)

print(f"\nAccuracy en test: {accuracy_score(y_test, y_pred):.4f}")
print("\nReporte de clasificación en test:")
print(classification_report(y_test, y_pred))

Accuracy por fold (10 K-Folds): [0.7258 0.7742 0.6613 0.7742 0.7377 0.8033 0.7705 0.8197 0.7213 0.7705]
Accuracy media en CV: 0.7558 (+/- 0.0433)

Accuracy en test: 0.7468

Reporte de clasificación en test:
              precision    recall  f1-score   support

           0       0.79      0.83      0.81       100
           1       0.65      0.59      0.62        54

    accuracy                           0.75       154
   macro avg       0.72      0.71      0.72       154
weighted avg       0.74      0.75      0.74       154



### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [53]:
rf_clf = RandomForestClassifier(
    n_estimators=100, max_features=3, random_state=42
)

scores = cross_val_score(rf_clf, X_train, y_train, cv=kfold, scoring="accuracy")

print(f"Accuracy media CV (10 folds): {scores.mean():.4f}")

rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.4f}")

Accuracy media CV (10 folds): 0.7590
Accuracy en test: 0.7597


In [37]:
importancias = pd.Series(rf_clf.feature_importances_, index=X.columns)
print(importancias.sort_values(ascending=False))

plas    0.292768
mass    0.168607
pedi    0.124303
age     0.123728
pres    0.088036
preg    0.078773
test    0.062336
skin    0.061447
dtype: float64


### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [58]:
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=30,
    random_state=42,
)

scores = cross_val_score(ada_clf, X_train, y_train, cv=kfold, scoring="accuracy")

print(f"Accuracy media CV (10 folds): {scores.mean():.4f}")

ada_clf.fit(X_train, y_train)
y_pred = ada_clf.predict(X_test)
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.4f}")

Accuracy media CV (10 folds): 0.7511
Accuracy en test: 0.7857


### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [55]:
gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)

scores = cross_val_score(gb_clf, X_train, y_train, cv=kfold, scoring="accuracy")

print(f"Accuracy media CV (10 folds): {scores.mean():.4f}")

gb_clf.fit(X_train, y_train)
y_pred = gb_clf.predict(X_test)
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.4f}")

Accuracy media CV (10 folds): 0.7411
Accuracy en test: 0.7532


### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [40]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
xgb_clf = XGBClassifier(
    n_estimators=100,
    random_state=42,
    device= 'cuda', #para usar la grafica nvidia
    eval_metric="logloss",
)

scores = cross_val_score(xgb_clf, X_train, y_train, cv=kfold, scoring="accuracy")

print(f"Accuracy media CV (10 folds): {scores.mean():.4f}")

xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)
print(f"Accuracy en test: {accuracy_score(y_test, y_pred):.4f}")

c:\Users\haize\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:751: UserWarning: [10:58:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Accuracy media CV (10 folds): 0.7363
Accuracy en test: 0.7597


### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [67]:
modelos = {
    "Bagging (100)": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100,
        random_state=42,
    ),
    "Random Forest (100, max_features=3)": RandomForestClassifier(
        n_estimators=100,
        max_features=3,
        random_state=42,
    ),
    "AdaBoost (30)": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=30,
        random_state=42,
    ),
    "Gradient Boosting (100)": GradientBoostingClassifier(
        n_estimators=100,
        random_state=42,
    ),
    "XGBoost (100)": XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
    ),
}

filas = []

for nombre, modelo in modelos.items():
    scores = cross_val_score(
        modelo, X_train, y_train, cv=kfold, scoring="accuracy"
    )
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    filas.append({
        "Algoritmo": nombre,
        "CV Accuracy media": round(scores.mean(), 4),
        "CV Std": round(scores.std(), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred), 4),
    })

df_ordenado = (
    pd.DataFrame(filas)
    .sort_values(by="Test Accuracy", ascending=False)
    .reset_index(drop=True)
)

print(df_ordenado.to_string(index=False))

                          Algoritmo  CV Accuracy media  CV Std  Test Accuracy
                      AdaBoost (30)             0.7511  0.0597         0.7857
Random Forest (100, max_features=3)             0.7590  0.0511         0.7597
            Gradient Boosting (100)             0.7411  0.0584         0.7532
                      Bagging (100)             0.7558  0.0433         0.7468
                      XGBoost (100)             0.7347  0.0544         0.7338


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [68]:

configuraciones = {
    "Bagging": (
        BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=42), random_state=42
        ),
        {
            "n_estimators": [30, 50, 100],
            "max_samples": [0.7, 1.0],
            "max_features": [0.7, 1.0],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [50, 100, 150],
            "max_depth": [3, 5, None],
            "max_features": [2, 3, "sqrt"],
        },
    ),
    "AdaBoost": (
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(random_state=42), random_state=42
        ),
        {
            "estimator__max_depth": [1, 2],
            "n_estimators": [30, 50, 100],
            "learning_rate": [0.05, 0.1, 0.5, 1.0],
        },
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "n_estimators": [50, 100],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [2, 3, 4],
        },
    ),
    "XGBoost": (
        XGBClassifier(random_state=42, eval_metric="logloss"),
        {
            "n_estimators": [50, 100],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [2, 3, 4],
        },
    ),
}

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
filas = []

for nombre, (estimador, grid) in configuraciones.items():
    gs = GridSearchCV(
        estimator=estimador,
        param_grid=grid,
        cv=kfold,
        scoring="accuracy",
        n_jobs=-1,
    )
    gs.fit(X_train, y_train)

    y_pred = gs.best_estimator_.predict(X_test)
    std_mejor = gs.cv_results_["std_test_score"][gs.best_index_]

    filas.append({
        "Algoritmo": nombre,
        "CV Accuracy media": round(gs.best_score_, 4),
        "CV Std": round(std_mejor, 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred), 4),
    })

df_gs_ordenado = (
    pd.DataFrame(filas)
    .sort_values(by="Test Accuracy", ascending=False)
    .reset_index(drop=True)
)

print(df_gs_ordenado.to_string(index=False))

        Algoritmo  CV Accuracy media  CV Std  Test Accuracy
          Bagging             0.7705  0.0445         0.7532
    Random Forest             0.7704  0.0531         0.7532
         AdaBoost             0.7819  0.0602         0.7468
          XGBoost             0.7737  0.0589         0.7468
Gradient Boosting             0.7688  0.0599         0.7273


### 10. Conclusiones finales